In [ ]:
import numpy as np 
import pandas as pd
from pathlib import Path

In [ ]:
class DataCleaner:
    def __init__(self, filepath: Path = Path(r'..\datasets\Nassau Candy Distributor.csv')):
        self.filepath = filepath
        self.file = self.__get_data()

    def __get_data(self):
        try:
            return pd.read_csv(self.filepath, index_col=False)
        except Exception as e:
            print('Can not extract dataframe.', e)
    
    @staticmethod
    def validate_sales_column(x) -> bool:
        return round(x['Gross Profit'] + x['Cost'], 4) == round(x['Sales'], 4)

    def clean_and_validate_dataframe(self):
        file = self.file
        try:
            print('Data Cleaning in process....')
            print('-'*60)

            null_values = file.isna().sum().sum()
            if null_values > 0:
                print(f"{null_values} null values found. Dropping them...")
                file.dropna(inplace=True)
            else:
                print("No null values found.")
            
            assert file.isna().sum().sum() == 0, "Dataset still contains null values after cleaning"
            print('-'*60)
            
            duplicates = file.duplicated().sum()
            if duplicates > 0:
                print(f"{duplicates} duplicate values found. Dropping them...")
                file.drop_duplicates(keep='first', inplace=True)
            else:
                print("No duplicate values found.")
                
            assert file.duplicated().sum() == 0, "Dataset still contains duplicate values after cleaning"
            print('-'*60)

            file['is_sales_valid'] = file.apply(DataCleaner.validate_sales_column, axis=1)
            not_valid_rows = len(file[file['is_sales_valid'] == False])
            
            assert not_valid_rows == 0, f"Sales column validation failed. {not_valid_rows} invalid rows found."
            
            print('Sales column is validated successfully')
            file.drop(columns=['is_sales_valid'], inplace=True) 
            print('-'*60)

        except Exception as e:
            print('Exception Occured while cleaning and validating dataframe', e)
        
        self.cleaned_file = file
        return file


    def save_file(self, filename: Path, cleaned_file: pd.DataFrame | None = None):
        try:
            df_to_save = cleaned_file if cleaned_file is not None else self.cleaned_file
            df_to_save.to_csv(filename, index=False)
            print(f'Cleaned Dataframe saved successfully in {filename}')
        except Exception as e:
            print('Exception Occurred while saving the cleaned file.', e)

In [9]:
data_cleaner = DataCleaner()
cleaned_dataset = data_cleaner.clean_and_validate_dataframe() # clean the file
data_cleaner.save_file(filename=Path(r'..\datasets\cleaned_dataset.csv'), cleaned_file=cleaned_dataset) # saving the file

Data Cleaning in process....
------------------------------------------------------------
No null values found.
------------------------------------------------------------
No duplicate values found.
------------------------------------------------------------
Sales column is validated successfully
------------------------------------------------------------
Cleaned Dataframe saved successfully in ..\datasets\cleaned_dataset.csv
